# Qwen2.5-7B-Instruct Naked Test

Minimal notebook for direct generation without retriever, CCS, reranker, QUR, or verifier.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:
# Naked test: only the user question.
question = 'based on NIST SP 800-53, How many times can a user enter the wrong password before we lock their account?'

messages = [
    {'role': 'user', 'content': question}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors='pt').to(model.device)


In [5]:
with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False,
    )

new_tokens = generated_ids[:, model_inputs.input_ids.shape[1]:]
answer = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]

print('QUESTION:\n', question)
print('\nANSWER:\n', answer)


QUESTION:
 based on NIST SP 800-53, How many times can a user enter the wrong password before we lock their account?

ANSWER:
 NIST Special Publication (SP) 800-53 does not specify a specific number of failed login attempts before an account should be locked. Instead, it provides guidelines and recommendations for security controls that organizations can implement to protect against unauthorized access.

According to NIST SP 800-53, the number of failed login attempts before locking an account is typically determined by the organization's risk management strategy and security policies. The publication emphasizes the importance of balancing security with usability and recommends implementing controls such as:

1. **Account Lockout**: This control can be configured to lock an account after a specified number of failed login attempts. The number of attempts can vary based on the sensitivity of the system and the organization's risk tolerance. For example, a less sensitive system might all